## Разложить $x^7 - 1$ на множители (должно быть 3) над полем $F_2$

Очевидно у многочлена имеется корень x = 1. Тогда:

$$
x^7 - 1 = (x + 1)(x^6 + x^5 + x^4 + x^3 + x^2 + x + 1)
$$
Дальше можно разложить как:
$$
x^7 - 1 = (x + 1)(x^3 + x + 1)(x^3 + x^2 + 1)
$$

Проверка:
- $(x^3 + x + 1)(x^3 + x^2 + 1) = x^6 + x^5 + x^4 + x^3 + x^2 + x + 1$

Неприводимые полиномы:
- $p_0(x) = x + 1$
- $p_1(x) = x^3 + x + 1$
- $p_2(x) = x^3 + x^2 + 1$

## Построить из их комбинаций $6$ порождающих полиномов

Все делители $x^7 - 1$ (кроме 1 и $x^7-1$) дают 6 нетривиальных циклических кодов.

| № | Порождающий многочлен $g(x)$ | Степень | Размерность $k = 7 - \deg g$ |
|---|------------------------------|---------|------------------------------|
| 1 | $x + 1$ | 1 | 6 |
| 2 | $x^3 + x + 1$ | 3 | 4 |
| 3 | $x^3 + x^2 + 1$ | 3 | 4 |
| 4 | $(x+1)(x^3+x+1) = x^4 + x^3 + x^2 + 1$ | 4 | 3 |
| 5 | $(x+1)(x^3+x^2+1) = x^4 + x^3 + x + 1$ | 4 | 3 |
| 6 | $(x^3+x+1)(x^3+x^2+1) = x^6 + x^5 + x^4 + x^3 + x^2 + x + 1$ | 6 | 1 |

In [4]:
import numpy as np
from itertools import product

def poly_to_vector(poly_coeffs, length=7):
    """
    Преобразует многочлен в двоичный вектор длины length.
    poly_coeffs: список коэффициентов от младшей степени к старшей
    """
    vec = np.zeros(length, dtype=int)
    for i, coeff in enumerate(poly_coeffs):
        if i < length:
            vec[i] = coeff % 2
    return vec

def cyclic_shift_left(vec, shift=1):
    """Циклический сдвиг влево"""
    return np.roll(vec, -shift)

def generate_code_words(generator_poly_coeffs, n=7):
    """
    Генерирует все кодовые слова циклического кода с порождающим многочленом g(x)
    generator_poly_coeffs: коэффициенты g(x) от младшей степени к старшей
    """
    g_vec = poly_to_vector(generator_poly_coeffs, n)
    k = n - (len(generator_poly_coeffs) - 1)  # степень g = len-1, размерность k = n - deg(g)

    if k <= 0:
        return [np.zeros(n, dtype=int)]

    # Все возможные информационные слова (длины k)
    code_words = []
    for info_bits in product([0, 1], repeat=k):
        # Cгенерируем все возможные сдвиги порождающего многочлена и их суммы
        # базисные векторы — циклические сдвиги g(x)
        basis = []
        current = g_vec.copy()
        for _ in range(k):
            basis.append(current.copy())
            current = cyclic_shift_left(current)

        # Кодовое слово = сумма базисных векторов по информационным битам
        codeword = np.zeros(n, dtype=int)
        for j, bit in enumerate(info_bits):
            if bit:
                codeword = (codeword + basis[j]) % 2
        code_words.append(codeword)

    return code_words

def weight(vec):
    """Хэммингов вес"""
    return np.sum(vec)

def analyze_code(name, generator_coeffs, n=7):
    """Анализирует код: размерность, минимальный вес, распределение весов"""
    code_words = generate_code_words(generator_coeffs, n)
    k = len(generator_coeffs) - 1
    k = n - k  # размерность

    weights = [weight(cw) for cw in code_words]
    min_weight = min(w for w in weights if w > 0)
    max_weight = max(weights)

    # Подсчёт распределения весов
    from collections import Counter
    weight_dist = Counter(weights)

    print(f"\n{'='*60}")
    print(name)
    print(f"Порождающий многочлен: {generator_coeffs}")
    print(f"Параметры: [7, {k}, {min_weight}]")
    print(f"Количество кодовых слов: {len(code_words)}")
    print(f"Распределение весов: {dict(sorted(weight_dist.items()))}")

    return weights


# g1 = x + 1
g1 = [1, 1]  # 1 + x

# g2 = x^3 + x + 1
g2 = [1, 1, 0, 1]  # 1 + x + x^3

# g3 = x^3 + x^2 + 1
g3 = [1, 0, 1, 1]  # 1 + x^2 + x^3

# g4 = (x+1)(x^3+x+1) = x^4 + x^3 + x^2 + 1
g4 = [1, 0, 1, 1, 1]  # 1 + x^2 + x^3 + x^4

# g5 = (x+1)(x^3+x^2+1) = x^4 + x^3 + x + 1
g5 = [1, 1, 0, 1, 1]  # 1 + x + x^3 + x^4

# g6 = (x^3+x+1)(x^3+x^2+1) = x^6 + x^5 + x^4 + x^3 + x^2 + x + 1
g6 = [1, 1, 1, 1, 1, 1, 1]  # 1 + x + x^2 + x^3 + x^4 + x^5 + x^6

codes = [
    ("Код 1: x+1", g1),
    ("Код 2: x^3+x+1", g2),
    ("Код 3: x^3+x^2+1", g3),
    ("Код 4: (x+1)(x^3+x+1)", g4),
    ("Код 5: (x+1)(x^3+x^2+1)", g5),
    ("Код 6: (x^3+x+1)(x^3+x^2+1)", g6),
]

all_weights = {}
for name, gen in codes:
    weights = analyze_code(name, gen)
    all_weights[name] = weights


Код 1: x+1
Порождающий многочлен: [1, 1]
Параметры: [7, 6, 2]
Количество кодовых слов: 64
Распределение весов: {np.int64(0): 1, np.int64(2): 21, np.int64(4): 35, np.int64(6): 7}

Код 2: x^3+x+1
Порождающий многочлен: [1, 1, 0, 1]
Параметры: [7, 4, 3]
Количество кодовых слов: 16
Распределение весов: {np.int64(0): 1, np.int64(3): 7, np.int64(4): 7, np.int64(7): 1}

Код 3: x^3+x^2+1
Порождающий многочлен: [1, 0, 1, 1]
Параметры: [7, 4, 3]
Количество кодовых слов: 16
Распределение весов: {np.int64(0): 1, np.int64(3): 7, np.int64(4): 7, np.int64(7): 1}

Код 4: (x+1)(x^3+x+1)
Порождающий многочлен: [1, 0, 1, 1, 1]
Параметры: [7, 3, 4]
Количество кодовых слов: 8
Распределение весов: {np.int64(0): 1, np.int64(4): 7}

Код 5: (x+1)(x^3+x^2+1)
Порождающий многочлен: [1, 1, 0, 1, 1]
Параметры: [7, 3, 2]
Количество кодовых слов: 8
Распределение весов: {np.int64(0): 1, np.int64(2): 1, np.int64(4): 5, np.int64(6): 1}

Код 6: (x^3+x+1)(x^3+x^2+1)
Порождающий многочлен: [1, 1, 1, 1, 1, 1, 1]
Параметры

## Определить характеристики полученных кодов, определить, что это за коды?

| № | Код $[n, k, d]$ | Тип кода | Пояснение |
|---|----------------|----------|-----------|
| 1 | $[7, 6, 2]$ | Код с проверкой на чётность | Все кодовые слова имеют чётный вес |
| 2 | $[7, 4, 3]$ | Код Хэмминга | Исправляет одну ошибку |
| 3 | $[7, 4, 3]$ | Код Хэмминга | Эквивалентен коду №2 (перестановка координат) |
| 4 | $[7, 3, 4]$ | Симплекс-код | Двойственный к коду Хэмминга; все ненулевые слова имеют вес 4 |
| 5 | $[7, 3, 4]$ | Симплекс-код | Эквивалентен коду №4 |
| 6 | $[7, 1, 7]$ | Код с повторением | Только слова $(0000000)$ и $(1111111)$ |